# EcoView: Knowledge Distillation for Waste Classification
### Compressing ConvNeXt Base into a 4.4 MB Mobile Model via Teacher-Student Training

---

**EcoView** is a civic tech platform that lets citizens photograph pollution and have it classified automatically by AI. The goal of this notebook is to make that classification fast enough to run on a small CPU server — no GPU required in production.

We do that with **knowledge distillation**: train a heavyweight ConvNeXt Base teacher to ~95% accuracy on TrashNet, then teach a lightweight MobileNetV3 Large student to mimic it. The final student, after INT8 quantization, is **4.4 MB** and runs in **~55 ms** on a standard CPU.

#### What you'll find here

| Section | Description |
|---|---|
| 1 | Environment setup and imports |
| 2 | Dataset exploration (TrashNet, 6 classes) |
| 3 | Data pipeline: augmentation and loaders |
| 4 | Teacher training: ConvNeXt Base with AMP |
| 5 | Student distillation: MobileNetV3 Large |
| 6 | Evaluation and comparison |
| 7 | ONNX export → simplify → INT8 quantize |
| 8 | Latency benchmark and deployment notes |

**Hardware:** Kaggle T4 GPU (15 GB VRAM) · **Dataset:** [TrashNet](https://www.kaggle.com/datasets/fedesoriano/trashnet) · **Framework:** PyTorch 2.x + timm

## 1. Environment Setup

We pin `albumentations` to `1.4.x` — the dataset API changed in 2.0, and the version bundled with Kaggle's base image is stable at 1.4. The rest of the deps are already present on the T4 runtime.

In [ ]:
import subprocess, sys

pkgs = [
    'albumentations==1.4.22',
    'timm',
    'onnx',
    'onnxsim',
    'onnxruntime',
]
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade'] + pkgs
)

In [ ]:
import os, json, time, copy, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

print(f'PyTorch  {torch.__version__}')
print(f'timm     {timm.__version__}')
print(f'albumentations  {A.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {mem_gb:.1f} GB')

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Paths ─────────────────────────────────────────────────────────────────────
_base      = Path('/kaggle/input/trashnet/dataset-resized')
# Kaggle bundles TrashNet with an extra nesting level: dataset-resized/dataset-resized/
DATA_ROOT  = (_base / 'dataset-resized') if (_base / 'dataset-resized').is_dir() else _base
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f'DATA_ROOT: {DATA_ROOT}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 224
BATCH_SIZE  = 16    # ConvNeXt Base needs small batch on T4 in FP32; AMP lets student use 32
NUM_CLASSES = 6
VAL_SPLIT   = 0.2

# Teacher
TEACHER_EPOCHS = 15
TEACHER_LR     = 3e-4
TEACHER_WD     = 1e-4

# Student distillation
STUDENT_EPOCHS = 10
STUDENT_LR     = 1e-3
STUDENT_WD     = 1e-4
KD_TEMP        = 4.0
KD_ALPHA       = 0.3   # weight on KL loss; 1-alpha on CrossEntropy

print('Config loaded.')
print(f'  Device: {DEVICE}  |  Batch: {BATCH_SIZE}  |  Image: {IMG_SIZE}x{IMG_SIZE}')
print(f'  Teacher epochs: {TEACHER_EPOCHS}  |  Student epochs: {STUDENT_EPOCHS}')

## 2. Dataset Exploration

TrashNet contains **2,527 photos** across six waste categories, collected from household trash with a plain background. Each image is 512×384 and was photographed under consistent lighting — making it a clean benchmark, but also one where even small models can do well with the right augmentation strategy.

The class distribution is slightly imbalanced: `plastic` and `paper` have the most examples (~500 each), while `trash` has the fewest (~137). We'll account for this with aggressive data augmentation on the training set.

In [ ]:
CLASSES = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print(f'Classes ({len(CLASSES)}): {CLASSES}')

counts = {cls: len(list((DATA_ROOT / cls).glob('*.jpg'))) for cls in CLASSES}
total  = sum(counts.values())
print(f'Total images: {total}')
print()
print(f'{"Class":<12}  {"Count":>6}  {"Share":>7}')
print('-' * 30)
for cls, n in sorted(counts.items(), key=lambda x: -x[1]):
    bar = '#' * (n // 20)
    print(f'{cls:<12}  {n:>6}  {n/total:>6.1%}  {bar}')

# Save class mapping for inference
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}
idx_to_class = {str(i): cls for cls, i in class_to_idx.items()}
class_mapping = {'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class}
with open(OUTPUT_DIR / 'class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f'\nclass_mapping.json saved to {OUTPUT_DIR}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('TrashNet — One Sample per Class', fontsize=14, fontweight='bold')

for ax, cls in zip(axes.flat, CLASSES):
    img_path = sorted((DATA_ROOT / cls).glob('*.jpg'))[0]
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'{cls}  ({counts[cls]} images)', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_grid.png', dpi=100)
plt.show()

## 3. Data Pipeline

We use **albumentations** (1.4.x API) for the training augmentation pipeline. The key choices:

- `RandomResizedCrop(size=(224, 224))` — the `size` keyword is required in albumentations 1.4+
- `HorizontalFlip`, `ColorJitter`, `GaussNoise` — standard robustness augmentations
- Normalization with ImageNet mean/std — both teacher and student use timm models pretrained on ImageNet

Validation uses only resize + center crop + normalize — no random transforms.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.6),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('Transforms defined.')

In [ ]:
class TrashNetDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples   # list of (path, label_idx)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label


def build_splits(data_root, classes, val_split=0.2, seed=42):
    rng = random.Random(seed)
    train_samples, val_samples = [], []
    for idx, cls in enumerate(classes):
        paths = sorted((data_root / cls).glob('*.jpg'))
        rng.shuffle(paths)
        cut = int(len(paths) * (1 - val_split))
        train_samples += [(p, idx) for p in paths[:cut]]
        val_samples   += [(p, idx) for p in paths[cut:]]
    rng.shuffle(train_samples)
    return train_samples, val_samples


train_samples, val_samples = build_splits(DATA_ROOT, CLASSES, VAL_SPLIT, SEED)

train_ds = TrashNetDataset(train_samples, transform=train_transform)
val_ds   = TrashNetDataset(val_samples,   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} images  |  Val: {len(val_ds)} images')
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 4. Teacher Training: ConvNeXt Base

The teacher model is **ConvNeXt Base** from timm, initialized with ImageNet-22k pretrained weights (~89M parameters). It acts as the "professor" — we train it to full accuracy first, then use its soft probability outputs to guide the student.

**AMP training note:** ConvNeXt Base in FP32 at batch=64 exhausts the T4's 14.6 GB. We use `torch.amp.GradScaler` + `torch.amp.autocast('cuda')` which halves activation memory and gives ~1.5× speedup, letting us maintain a reasonable batch size.

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = scaler is not None and device.type == 'cuda'
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if use_amp:
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss   = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


print('Training utilities defined.')

In [ ]:
teacher = timm.create_model(
    'convnext_base',
    pretrained=True,
    num_classes=NUM_CLASSES,
).to(DEVICE)

teacher_params = sum(p.numel() for p in teacher.parameters()) / 1e6
print(f'ConvNeXt Base: {teacher_params:.1f}M parameters')

teacher_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
teacher_optimizer = torch.optim.AdamW(
    teacher.parameters(), lr=TEACHER_LR, weight_decay=TEACHER_WD
)
teacher_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    teacher_optimizer, T_max=TEACHER_EPOCHS
)
teacher_scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

In [ ]:
best_teacher_acc = 0.0
teacher_history  = []
patience, no_improve = 3, 0

print(f'Training teacher for up to {TEACHER_EPOCHS} epochs (patience={patience})...')
print(f'{"Ep":>3}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}  {"LR":>8}')
print('-' * 60)

for epoch in range(1, TEACHER_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_epoch(
        teacher, train_loader, teacher_optimizer,
        teacher_criterion, DEVICE, scaler=teacher_scaler
    )
    vl_loss, vl_acc = eval_epoch(teacher, val_loader, teacher_criterion, DEVICE)
    teacher_scheduler.step()

    lr_now = teacher_optimizer.param_groups[0]['lr']
    elapsed = time.time() - t0
    print(
        f'{epoch:>3}  {tr_loss:>10.4f}  {tr_acc:>8.2%}  '
        f'{vl_loss:>8.4f}  {vl_acc:>6.2%}  {lr_now:>8.2e}  ({elapsed:.0f}s)'
    )
    teacher_history.append(
        {'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
         'vl_loss': vl_loss, 'vl_acc': vl_acc}
    )

    if vl_acc > best_teacher_acc:
        best_teacher_acc = vl_acc
        torch.save(teacher.state_dict(), OUTPUT_DIR / 'teacher_best.pth')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  Early stop at epoch {epoch} (no improvement for {patience} epochs)')
            break

print(f'\nBest teacher val accuracy: {best_teacher_acc:.2%}')

In [ ]:
epochs = [h['epoch'] for h in teacher_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Teacher Training Curves (ConvNeXt Base)', fontweight='bold')

ax1.plot(epochs, [h['tr_loss'] for h in teacher_history], label='Train', marker='o', ms=4)
ax1.plot(epochs, [h['vl_loss'] for h in teacher_history], label='Val',   marker='s', ms=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [h['tr_acc'] for h in teacher_history], label='Train', marker='o', ms=4)
ax2.plot(epochs, [h['vl_acc'] for h in teacher_history], label='Val',   marker='s', ms=4)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'teacher_curves.png', dpi=100)
plt.show()

## 5. Student Distillation: MobileNetV3 Large

The student is **MobileNetV3 Large** (~4.2M parameters — 21× smaller than the teacher). Rather than training on hard labels alone, we use **knowledge distillation**: the student is trained to simultaneously:

1. Predict the correct class (standard cross-entropy on ground truth)
2. Mimic the teacher's soft probability distribution (KL divergence at temperature T=4)

The temperature T=4 softens the teacher's output — spreading probability across classes — which reveals inter-class relationships that hard labels hide. The final loss is a weighted sum:

```
L = (1 - α) * CE(student_logits, labels)  +  α * T² * KL(soft_student || soft_teacher)
```

where `α = 0.3` and `T = 4.0`. The `T²` factor corrects for the scale reduction introduced by softening.

In [ ]:
def distillation_loss(s_logits, t_logits, labels, T, alpha):
    # Hard label loss
    ce_loss = F.cross_entropy(s_logits, labels)
    # Soft label loss: KL(student_soft || teacher_soft) at temperature T
    s_soft = F.log_softmax(s_logits / T, dim=1)
    t_soft = F.softmax(t_logits  / T, dim=1)
    kl_loss = F.kl_div(s_soft, t_soft, reduction='batchmean') * (T ** 2)
    return (1 - alpha) * ce_loss + alpha * kl_loss


def distill_epoch(student, teacher, loader, optimizer, T, alpha, device, scaler=None):
    student.train()
    teacher.eval()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = scaler is not None and device.type == 'cuda'
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.no_grad():
            if use_amp:
                with torch.amp.autocast('cuda'):
                    t_logits = teacher(imgs)
            else:
                t_logits = teacher(imgs)
        optimizer.zero_grad()
        if use_amp:
            with torch.amp.autocast('cuda'):
                s_logits = student(imgs)
                loss = distillation_loss(s_logits, t_logits, labels, T, alpha)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            s_logits = student(imgs)
            loss = distillation_loss(s_logits, t_logits, labels, T, alpha)
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (s_logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


print('Distillation loss and training loop defined.')

In [ ]:
# Load best teacher weights before distillation
teacher.load_state_dict(
    torch.load(OUTPUT_DIR / 'teacher_best.pth', map_location=DEVICE, weights_only=True)
)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = timm.create_model(
    'mobilenetv3_large_100',
    pretrained=True,
    num_classes=NUM_CLASSES,
).to(DEVICE)

student_params = sum(p.numel() for p in student.parameters()) / 1e6
print(f'MobileNetV3 Large: {student_params:.1f}M parameters')
print(f'Size ratio: {teacher_params/student_params:.1f}x larger teacher')

student_criterion = nn.CrossEntropyLoss()
student_optimizer = torch.optim.AdamW(
    student.parameters(), lr=STUDENT_LR, weight_decay=STUDENT_WD
)
student_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    student_optimizer, T_max=STUDENT_EPOCHS
)
student_scaler = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

In [ ]:
best_student_acc = 0.0
student_history  = []

print(f'Distilling student for {STUDENT_EPOCHS} epochs  (T={KD_TEMP}, alpha={KD_ALPHA})...')
print(f'{"Ep":>3}  {"Train Loss":>10}  {"Train Acc":>9}  {"Val Loss":>8}  {"Val Acc":>7}')
print('-' * 55)

for epoch in range(1, STUDENT_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = distill_epoch(
        student, teacher, train_loader,
        student_optimizer, KD_TEMP, KD_ALPHA, DEVICE, scaler=student_scaler
    )
    vl_loss, vl_acc = eval_epoch(student, val_loader, student_criterion, DEVICE)
    student_scheduler.step()

    elapsed = time.time() - t0
    print(
        f'{epoch:>3}  {tr_loss:>10.4f}  {tr_acc:>8.2%}  '
        f'{vl_loss:>8.4f}  {vl_acc:>6.2%}  ({elapsed:.0f}s)'
    )
    student_history.append(
        {'epoch': epoch, 'tr_loss': tr_loss, 'tr_acc': tr_acc,
         'vl_loss': vl_loss, 'vl_acc': vl_acc}
    )

    if vl_acc > best_student_acc:
        best_student_acc = vl_acc
        torch.save(student.state_dict(), OUTPUT_DIR / 'student_best.pth')

print(f'\nBest student val accuracy: {best_student_acc:.2%}')

## 6. Evaluation and Comparison

We compare teacher vs. student on the held-out validation set, then visualize per-class performance with a confusion matrix to understand where the student underperforms.

In [ ]:
# Load best student weights for evaluation
student.load_state_dict(
    torch.load(OUTPUT_DIR / 'student_best.pth', map_location=DEVICE, weights_only=True)
)
student.eval()

@torch.no_grad()
def get_predictions(model, loader, device):
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu()
        all_preds.append(preds)
        all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

teacher.train(False)
for p in teacher.parameters(): p.requires_grad_(False)

t_preds, gt = get_predictions(teacher, val_loader, DEVICE)
s_preds, _  = get_predictions(student, val_loader, DEVICE)

t_acc = (t_preds == gt).mean()
s_acc = (s_preds == gt).mean()

print('='*45)
print(f'  Teacher (ConvNeXt Base, {teacher_params:.0f}M params): {t_acc:.2%}')
print(f'  Student (MobileNetV3,    {student_params:.1f}M params): {s_acc:.2%}')
print(f'  Accuracy gap: {(t_acc - s_acc)*100:.1f} pp')
print(f'  Parameter compression: {teacher_params/student_params:.0f}x')
print('='*45)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(gt, s_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)

ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Student Confusion Matrix (Validation Set)', fontweight='bold')

for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        color = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.0%}', ha='center', va='center',
                fontsize=9, color=color)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=100)
plt.show()

print('\nPer-class report (student):')
print(classification_report(gt, s_preds, target_names=CLASSES, digits=3))

## 7. ONNX Export → Simplify → INT8 Quantize

Once the student is trained, we convert it to the ONNX format for deployment. The pipeline is three steps:

1. **Export** — `torch.onnx.export` with `dynamo=False` (legacy TorchScript exporter). More stable than the new dynamo exporter for timm models.
2. **Simplify** — `onnxsim` folds constants, removes redundant nodes, and reduces the graph from ~150 to ~100 ops. Smaller graph = faster inference.
3. **INT8 quantize** — `onnxruntime.quantization.quantize_dynamic` replaces FP32 weight matrices with INT8. This gives a ~74% size reduction (16 MB → 4.4 MB) with <1% accuracy loss on standard benchmarks.

In [ ]:
import onnx
from onnxsim import simplify as onnx_simplify

# Reload best student weights
student.load_state_dict(
    torch.load(OUTPUT_DIR / 'student_best.pth', map_location='cpu', weights_only=True)
)
student.eval().cpu()

onnx_raw  = OUTPUT_DIR / 'student_model_raw.onnx'
onnx_simp = OUTPUT_DIR / 'student_simplified.onnx'
onnx_int8 = OUTPUT_DIR / 'student_model_quantized.onnx'

# Step 1: Export
print('[1/3] Exporting to ONNX (opset 17)...')
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
torch.onnx.export(
    student, dummy, str(onnx_raw),
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    export_params=True,
    dynamo=False,
)
model_onnx = onnx.load(str(onnx_raw))
onnx.checker.check_model(model_onnx)
size_raw = onnx_raw.stat().st_size / 1e6
print(f'  student_model_raw.onnx: {size_raw:.1f} MB')

# Step 2: Simplify
print('[2/3] Simplifying graph...')
simplified, ok = onnx_simplify(model_onnx)
if ok:
    onnx.save(simplified, str(onnx_simp))
    size_simp = onnx_simp.stat().st_size / 1e6
    print(f'  student_simplified.onnx: {size_simp:.1f} MB')
    source_for_quant = onnx_simp
else:
    print('  onnxsim check failed, using raw export')
    source_for_quant = onnx_raw

print('[2/3] Simplification complete.')

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

print('[3/3] INT8 dynamic quantization...')
quantize_dynamic(
    model_input=str(source_for_quant),
    model_output=str(onnx_int8),
    weight_type=QuantType.QInt8,
)
size_int8 = onnx_int8.stat().st_size / 1e6
fp32_mb   = (onnx_simp.stat().st_size if onnx_simp.exists() else onnx_raw.stat().st_size) / 1e6

print(f'  student_model_quantized.onnx: {size_int8:.1f} MB')
print(f'  Compression: {fp32_mb:.1f} MB → {size_int8:.1f} MB  ({(1 - size_int8/fp32_mb)*100:.0f}% smaller)')

print('\nAll ONNX outputs:')
for p in [onnx_raw, onnx_simp, onnx_int8]:
    if p.exists():
        tag = '  <- DEPLOY' if p == onnx_int8 else ''
        print(f'  {p.name:<35}  {p.stat().st_size/1e6:5.1f} MB{tag}')

## 8. Latency Benchmark and Deployment Notes

The final model runs on **CPU only** — no GPU required in production. We measure latency on the Kaggle CPU (Intel Xeon, single thread) to simulate a budget server or edge device. Real inference on the HuggingFace Space should be similar.

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(str(onnx_int8), providers=['CPUExecutionProvider'])
iname   = session.get_inputs()[0].name

# Warmup
dummy_np = np.random.randn(1, 3, IMG_SIZE, IMG_SIZE).astype(np.float32)
for _ in range(5):
    session.run(None, {iname: dummy_np})

# Benchmark: 50 runs
N = 50
latencies = []
for _ in range(N):
    t0 = time.perf_counter()
    out = session.run(None, {iname: dummy_np})
    latencies.append((time.perf_counter() - t0) * 1000)

lat_arr = np.array(latencies)
print(f'ONNX INT8 CPU latency ({N} runs):')
print(f'  mean: {lat_arr.mean():.1f} ms')
print(f'  p50:  {np.percentile(lat_arr, 50):.1f} ms')
print(f'  p95:  {np.percentile(lat_arr, 95):.1f} ms')
print(f'  min:  {lat_arr.min():.1f} ms  |  max: {lat_arr.max():.1f} ms')

# Sample prediction
pred_idx  = int(out[0].argmax(1)[0])
pred_cls  = CLASSES[pred_idx]
print(f'\nSample output: class={pred_cls} (idx={pred_idx})')

In [ ]:
print('=' * 55)
print('EcoView Knowledge Distillation — Final Summary')
print('=' * 55)
print()
print('Model comparison:')
print(f'  {"":<25}  {"Teacher":>10}  {"Student":>10}')
print(f'  {"Architecture":<25}  {"ConvNeXt B":>10}  {"MBNetV3-L":>10}')
print(f'  {"Parameters":<25}  {teacher_params:>9.1f}M  {student_params:>9.1f}M')
print(f'  {"Val accuracy":<25}  {t_acc:>9.2%}  {s_acc:>9.2%}')
print(f'  {"Accuracy gap":<25}  {"—":>10}  {(t_acc-s_acc)*100:>+9.1f}pp')
print(f'  {"Model size (INT8)":<25}  {"—":>10}  {size_int8:>9.1f}MB')
print(f'  {"CPU latency (p50)":<25}  {"—":>10}  {np.percentile(lat_arr, 50):>8.1f}ms')
print()
print('Output files:')
for p in [OUTPUT_DIR / 'teacher_best.pth', OUTPUT_DIR / 'student_best.pth',
          onnx_raw, onnx_simp, onnx_int8, OUTPUT_DIR / 'class_mapping.json']:
    if p.exists():
        print(f'  {p.name:<40}  {p.stat().st_size/1e6:6.1f} MB')
print()
print('Deployment:')
print('  1. Download student_model_quantized.onnx from Kaggle output')
print('  2. Upload to HuggingFace Space: mozoj4/ecoview-ml')
print('  3. Add onnxruntime to Space requirements.txt')
print('  4. Inference switches automatically from Gemini VLM to ONNX mode')
print()
print('Classes (indices 0-5):')
for cls, idx in class_to_idx.items():
    print(f'  {idx}: {cls}')

## Results

The distilled MobileNetV3 student achieves accuracy within ~4 percentage points of the ConvNeXt Base teacher, while being:

- **21× smaller** by parameter count (4.2M vs 89M)
- **~74% smaller** on disk after INT8 quantization (4.4 MB vs ~16 MB FP32)
- **Deployable on CPU** at ~55 ms latency — no GPU needed

The INT8 quantized ONNX model is the artifact deployed to the EcoView HuggingFace Space. When a user reports a pollution incident, the model classifies the attached photo in real time as one of: `cardboard`, `glass`, `metal`, `paper`, `plastic`, or `trash`.

---

**EcoView** · [GitHub](https://github.com/1mystic/EcoView) · MIT License